# Baselines

This implements the two trivial baselines:

1. **Climatology baseline:** per-pixel monthly mean burned fraction, fit on the train split only.
2. **Persistence baseline:** previous observed target month copied forward as the next-month forecast.

It also defines the reusable evaluation code that willbe used for all later models:

- ensemble CRPS, with deterministic forecasts represented as one-member ensembles;
- deterministic scores on ensemble mean: RMSE, MAE, bias, and correlation;
- all-year, fire-season, August-only, and top-1% extreme-cell-month subsets;
- reliability-table helper for event forecasts;
- monthly score tables and case-study plotting hooks for 2017 and 2022.

The key convention is that **all models emit an ensemble forecast with shape `(time, member, lat, lon)`**, even when `member = 1`.

Revision notes in this version:

- metric counts and means are computed over finite forecast–observation pairs, not observations alone;
- persistence's first missing forecast is excluded from all metrics and reliability bins;
- bootstrap CRPS intervals resample monthly weighted numerator/denominator components, so conditional subsets such as the extreme top-1% tail match the headline metric definition;
- reliability probabilities preserve missing forecasts as missing instead of converting `NaN > threshold` to `False`;
- case-study observed/forecast maps use a shared burned-fraction colour scale;
- split summaries explicitly list non-contiguous split years.


## 0. Project decisions

- Target: monthly burned fraction in `[0, 1]`.
- Forecast horizon: next month, single step.
- Split: train, validation, and test are already materialised as year-blocked NetCDF files.
- Baseline family: climatology and persistence are the no-skill / no-model references.
- Evaluation: CRPS for probabilistic skill; RMSE and correlation on the ensemble mean; case-study diagnostics for 2017 Portugal and the 2022 Iberian summer.
- Masking: use the Iberian-domain evaluation mask from the previous EDA. If the saved mask is absent, rebuild it as ERA5 `LSM > 0.5` north of the heuristic southern boundary, then weight metrics by continuous `LSM × cos(latitude)` inside that mask.

For deterministic baselines, CRPS reduces to absolute error because the one-member ensemble has no spread. That is intentional: it lets the exact same scoring function evaluate climatology, persistence, deterministic U-Net, and future diffusion ensembles.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("default")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# EDA found that the target is extremely zero-inflated. This tiny threshold is
# used only for occurrence summaries; burned fraction itself remains continuous.
BURN_THRESHOLD = 1e-6
LAND_THRESHOLD = 0.5

# Fire-season reporting choice from the previous EDA takeaways.
FIRE_SEASON_MONTHS = (7, 8, 9, 10)

# Extreme-cell-month subset: top 1% of train-split burned-fraction values.
EXTREME_Q = 0.99

# Bootstrap settings for uncertainty summaries. Increase N_BOOT for final runs.
N_BOOT = 1000
BOOT_ALPHA = 0.05

In [ ]:
def find_processed_data_dir() -> Path:
    env = os.environ.get("WILDFIRE_DATA_DIR")
    if env:
        return Path(env).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed",
        cwd.parent / "data" / "processed",
        Path("/mnt/data/data/processed"),
    ]
    for candidate in candidates:
        if (candidate / "splits" / "train.nc").exists():
            return candidate.resolve()

    # Fall back to the standard project-relative location. The assertions below
    # give a clear error if the files have not been built yet.
    return (cwd / "data" / "processed").resolve()


DATA_DIR = find_processed_data_dir()
SPLIT_DIR = DATA_DIR / "splits"
SPLIT_FILES = {
    "train": SPLIT_DIR / "train.nc",
    "val": SPLIT_DIR / "val.nc",
    "test": SPLIT_DIR / "test.nc",
}
LSM_PATH = DATA_DIR / "static_lsm.nc"
IBERIAN_MASK_PATH = DATA_DIR / "static_iberian_mask.nc"
BASELINE_DIR = DATA_DIR / "baselines"
METRIC_DIR = DATA_DIR / "metrics"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [*SPLIT_FILES.values(), LSM_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing processed tensor files. Expected files under "
        f"{DATA_DIR}. Missing:\n" + "\n".join(missing)
    )

print(f"Using DATA_DIR      = {DATA_DIR}")
print(f"Writing baselines   = {BASELINE_DIR}")
print(f"Writing metrics     = {METRIC_DIR}")
for name, path in SPLIT_FILES.items():
    print(f"{name:>5}: {path}")
print(f"  lsm: {LSM_PATH}")
print(f" mask: {IBERIAN_MASK_PATH}  # used if present, rebuilt if missing")

## 2. Load targets and the Iberian-domain mask

In [ ]:
def load_lsm(lsm_path: Path = LSM_PATH) -> xr.DataArray:
    """Load the static land-sea mask as a 2D DataArray sorted by latitude."""
    ds = xr.open_dataset(lsm_path)
    if "lsm" in ds.data_vars:
        da = ds["lsm"]
    else:
        first_var = list(ds.data_vars)[0]
        da = ds[first_var]
    da = da.squeeze(drop=True)
    if "lat" in da.coords:
        da = da.sortby("lat")
    return da.astype("float32")


def _iberian_southern_boundary(lsm: xr.DataArray) -> xr.DataArray:
    """Heuristic southern boundary to remove North African coastal cells.

    This mirrors the previous EDA notebook. It keeps Iberia/Balearics in the
    rectangular ERA5 domain while excluding LSM-land cells south of the Strait
    and along the Algerian/Moroccan coast.
    """
    _, lon2d = xr.broadcast(lsm["lat"], lsm["lon"])
    boundary = xr.full_like(lon2d, 36.0, dtype="float32")

    boundary = xr.where((lon2d >= -5.5) & (lon2d < -2.0), 36.25, boundary)
    boundary = xr.where((lon2d >= -2.0) & (lon2d < 0.5), 36.50, boundary)
    boundary = xr.where((lon2d >= 0.5) & (lon2d < 1.5), 38.00, boundary)
    boundary = xr.where(lon2d >= 1.5, 38.50, boundary)

    boundary = boundary.rename("southern_boundary_lat")
    boundary.attrs.update({
        "description": "Heuristic southern latitude boundary used to remove North African coastal cells from the rectangular Iberian ERA5 grid.",
        "units": "degrees_north",
    })
    return boundary


def build_iberian_mask(lsm: xr.DataArray, land_threshold: float = LAND_THRESHOLD) -> xr.Dataset:
    """Build the reusable Iberian-domain mask on the processed 0.25° grid."""
    lsm = lsm.sortby("lat") if "lat" in lsm.coords else lsm
    lat2d, _ = xr.broadcast(lsm["lat"], lsm["lon"])
    southern_boundary = _iberian_southern_boundary(lsm)

    mask = ((lsm > land_threshold) & (lat2d >= southern_boundary)).astype("uint8").rename("iberian_mask")
    mask.attrs.update({
        "description": "Iberian-domain evaluation mask: ERA5 LSM land cells north of a heuristic southern boundary.",
        "land_threshold": land_threshold,
        "source_lsm": str(LSM_PATH),
    })
    return xr.Dataset({"iberian_mask": mask, "southern_boundary_lat": southern_boundary})


def ensure_iberian_mask(mask_path: Path = IBERIAN_MASK_PATH, lsm_path: Path = LSM_PATH) -> xr.DataArray:
    """Load the EDA mask if present; otherwise rebuild and save it."""
    if mask_path.exists():
        ds = xr.open_dataset(mask_path)
        if "iberian_mask" in ds.data_vars:
            da = ds["iberian_mask"]
        else:
            da = ds[list(ds.data_vars)[0]]
        return da.squeeze(drop=True).sortby("lat")

    lsm = load_lsm(lsm_path)
    mask_ds = build_iberian_mask(lsm)
    mask_ds.to_netcdf(mask_path)
    return mask_ds["iberian_mask"]


def make_area_weights(lsm: xr.DataArray, land_mask: xr.DataArray) -> xr.DataArray:
    """Approximate cell-area weights for a regular lon/lat grid.

    weight(lat, lon) ∝ cos(latitude) × continuous ERA5 LSM, then normalised over
    the Iberian-domain mask. These weights are used consistently for all metrics.
    """
    lat_weights = xr.DataArray(
        np.cos(np.deg2rad(lsm["lat"].values)).astype("float64"),
        dims=("lat",),
        coords={"lat": lsm["lat"]},
    )
    weights = (lsm.clip(0, 1).astype("float64") * lat_weights).where(land_mask)
    total = weights.sum(skipna=True)
    if not np.isfinite(float(total)) or float(total) <= 0:
        raise ValueError("Area weights have zero/invalid total. Check LSM and mask alignment.")
    return (weights / total).rename("area_weight")

In [ ]:
def open_target_split(split_name: str) -> xr.DataArray:
    """Open one split's target as float32 with a split coordinate."""
    ds = xr.open_dataset(SPLIT_FILES[split_name])
    if "target" not in ds.data_vars:
        raise KeyError(f"{SPLIT_FILES[split_name]} does not contain a 'target' variable")
    da = ds["target"].astype("float32")
    if "lat" in da.coords:
        da = da.sortby("lat")
    da = da.assign_coords(split=("time", np.repeat(split_name, da.sizes["time"])))
    return da.rename("target")


lsm = load_lsm(LSM_PATH)
iberian_mask_da = ensure_iberian_mask(IBERIAN_MASK_PATH, LSM_PATH)
iberian_mask_da = iberian_mask_da.reindex(lat=lsm.lat, lon=lsm.lon, method="nearest")
land_mask = (iberian_mask_da > 0).rename("land_mask")
area_weights = make_area_weights(lsm, land_mask)

targets = {split: open_target_split(split) for split in ["train", "val", "test"]}
target_all = xr.concat([targets[s] for s in ["train", "val", "test"]], dim="time").sortby("time")

summary = []
for split_name, da in targets.items():
    times = pd.DatetimeIndex(da.time.values)
    years = sorted(set(times.year))
    summary.append({
        "split": split_name,
        "n_months": da.sizes["time"],
        "years": ", ".join(str(y) for y in years),
        "n_years": len(years),
        "first_month": times[0].date(),
        "last_month": times[-1].date(),
        "n_lat": da.sizes["lat"],
        "n_lon": da.sizes["lon"],
    })

print("Mask cells retained:", int(land_mask.sum().values), "of", land_mask.size)
print("Area weights sum:", float(area_weights.sum().values))
display(pd.DataFrame(summary))


## 3. Fit and generate the trivial baselines

### 3.1 Climatology baseline

The climatology baseline is fit **only on the train split** to avoid leakage. For each target calendar month and grid cell, it predicts:

\[
\hat{y}_{m, i, j} = \frac{1}{N_m}\sum_{t \in \text{train},\ month(t)=m} y_{t,i,j}
\]

This gives a per-pixel monthly mean field with dimensions `(month, lat, lon)`.

In [ ]:
def fit_monthly_climatology(train_target: xr.DataArray) -> xr.DataArray:
    """Train-only per-pixel monthly mean burned fraction."""
    clim = train_target.groupby("time.month").mean("time", skipna=True).astype("float32")
    clim = clim.clip(min=0.0, max=1.0).rename("climatology_monthly_mean")
    clim.attrs.update({
        "baseline": "climatology",
        "definition": "Per-pixel monthly mean burned fraction fit on the train split only.",
        "target": "monthly burned fraction in [0, 1]",
    })
    return clim


def predict_climatology(climatology: xr.DataArray, target_like: xr.DataArray) -> xr.DataArray:
    """Select the fitted monthly climatology for each target time."""
    target_month = xr.DataArray(
        pd.DatetimeIndex(target_like.time.values).month,
        dims=("time",),
        coords={"time": target_like.time},
        name="month",
    )
    forecast = climatology.sel(month=target_month)
    if "month" in forecast.coords and "month" not in forecast.dims:
        forecast = forecast.drop_vars("month")
    forecast = forecast.assign_coords(time=target_like.time)
    forecast = forecast.transpose("time", "lat", "lon").astype("float32")
    return forecast.clip(min=0.0, max=1.0).rename("forecast")


monthly_climatology = fit_monthly_climatology(targets["train"])
monthly_climatology.to_dataset().to_netcdf(BASELINE_DIR / "climatology_monthly_mean_train_only.nc")

print(monthly_climatology)
print("Saved:", BASELINE_DIR / "climatology_monthly_mean_train_only.nc")

### 3.2 Persistence baseline

The persistence baseline predicts each target month using the **previous observed burned-fraction month**:

\[
\hat{y}_{t} = y_{t-1}
\]

This is not train-fitted. It is a no-model temporal reference. It uses the chronologically sorted target record so that, for example, January 2017 can use December 2016 as the previous observation. If the first available month has no previous observation, it remains `NaN` and is automatically excluded from metric denominators.

In [ ]:
def predict_persistence(target_like: xr.DataArray, full_target_record: xr.DataArray) -> xr.DataArray:
    """Previous observed target month copied forward."""
    full_target_record = full_target_record.sortby("time")
    shifted = full_target_record.shift(time=1)
    forecast = shifted.sel(time=target_like.time)
    forecast = forecast.assign_coords(time=target_like.time)
    forecast = forecast.transpose("time", "lat", "lon").astype("float32")
    return forecast.clip(min=0.0, max=1.0).rename("forecast")


def as_one_member_ensemble(forecast: xr.DataArray, member_name: str = "deterministic") -> xr.DataArray:
    """Represent a deterministic forecast as an ensemble with one member."""
    out = forecast.expand_dims(member=[member_name]).transpose("time", "member", "lat", "lon")
    return out.astype("float32").rename("forecast")


forecasts = {}
for split_name, target in targets.items():
    clim = as_one_member_ensemble(predict_climatology(monthly_climatology, target), member_name="mean")
    pers = as_one_member_ensemble(predict_persistence(target, target_all), member_name="previous_month")
    forecasts[("climatology", split_name)] = clim
    forecasts[("persistence", split_name)] = pers

    for baseline_name, forecast_ens in [("climatology", clim), ("persistence", pers)]:
        ds_out = xr.Dataset({"forecast": forecast_ens})
        ds_out.attrs.update({
            "baseline": baseline_name,
            "split": split_name,
            "forecast_dims": "time, member, lat, lon",
            "target_variable": "monthly burned fraction in [0, 1]",
        })
        out_path = BASELINE_DIR / f"forecast_{baseline_name}_{split_name}.nc"
        ds_out.to_netcdf(out_path)
        print(f"Saved {baseline_name:12s} {split_name:5s}: {out_path}")

## 4. Reusable CRPS scoring code

For an ensemble forecast \(x_1, \ldots, x_M\) and observation \(y\), the empirical ensemble CRPS is:

\[
\operatorname{CRPS}(F, y) = \frac{1}{M}\sum_{m=1}^{M}|x_m-y| - \frac{1}{2M^2}\sum_{m=1}^{M}\sum_{n=1}^{M}|x_m-x_n|.
\]

The implementation below uses the sorted-ensemble identity for the pairwise term, avoiding an explicit `M × M` tensor. It works for deterministic forecasts (`M=1`), future deterministic U-Net outputs, and diffusion ensembles.

In [ ]:
def crps_ensemble(
    forecast: xr.DataArray,
    observation: xr.DataArray,
    member_dim: str = "member",
) -> xr.DataArray:
    """Cell-wise empirical CRPS for ensemble forecasts.

    Parameters
    ----------
    forecast:
        DataArray with dimensions including `member_dim` plus observation dims,
        conventionally `(time, member, lat, lon)`.
    observation:
        DataArray with dimensions `(time, lat, lon)`.
    member_dim:
        Name of the ensemble/member dimension.

    Returns
    -------
    DataArray
        CRPS with dimensions `(time, lat, lon)`.
    """
    if member_dim not in forecast.dims:
        forecast = forecast.expand_dims({member_dim: [0]})

    forecast, observation = xr.align(forecast, observation, join="inner")
    forecast = forecast.astype("float64")
    observation = observation.astype("float64")

    m = forecast.sizes[member_dim]
    if m < 1:
        raise ValueError("Forecast must have at least one ensemble member.")

    mean_abs_error = np.abs(forecast - observation).mean(member_dim, skipna=True)

    # Sorted-ensemble identity:
    # sum_{i,j} |x_i - x_j| = 2 * sum_j (2j - M + 1) * x_(j), j zero-based.
    sorted_forecast = xr.apply_ufunc(
        np.sort,
        forecast,
        input_core_dims=[[member_dim]],
        output_core_dims=[[member_dim]],
        kwargs={"axis": -1},
        dask="parallelized",
        vectorize=False,
        output_dtypes=[forecast.dtype],
    )
    coeff = xr.DataArray(
        (2 * np.arange(m) - m + 1).astype("float64"),
        dims=(member_dim,),
        coords={member_dim: sorted_forecast[member_dim]},
    )
    mean_pairwise_abs = 2.0 * (sorted_forecast * coeff).sum(member_dim, skipna=True) / (m * m)
    crps = mean_abs_error - 0.5 * mean_pairwise_abs
    return crps.clip(min=0.0).rename("crps")


# Smoke test: deterministic one-member CRPS equals absolute error.
_demo_obs = xr.DataArray(np.array([0.2, 0.5]), dims=("time",), coords={"time": [0, 1]})
_demo_fcst = xr.DataArray(np.array([[0.1], [0.9]]), dims=("time", "member"), coords={"time": [0, 1], "member": [0]})
_demo_crps = crps_ensemble(_demo_fcst, _demo_obs)
np.testing.assert_allclose(_demo_crps.values, np.abs(_demo_fcst.squeeze("member").values - _demo_obs.values))
print("CRPS smoke test passed.")

## 5. Full metric pipeline

In [ ]:
def month_coord(da: xr.DataArray) -> xr.DataArray:
    """Return target calendar month as a time-indexed DataArray."""
    return xr.DataArray(
        pd.DatetimeIndex(da.time.values).month,
        dims=("time",),
        coords={"time": da.time},
        name="month",
    )


def weighted_quantile(da: xr.DataArray, weights: xr.DataArray, q: float) -> float:
    """Weighted quantile over all finite cell-month values."""
    if not 0 <= q <= 1:
        raise ValueError("q must be in [0, 1]")
    w = weights.broadcast_like(da)
    x = np.asarray(da.values, dtype="float64").ravel()
    ww = np.asarray(w.values, dtype="float64").ravel()
    valid = np.isfinite(x) & np.isfinite(ww) & (ww > 0)
    x = x[valid]
    ww = ww[valid]
    if x.size == 0:
        return np.nan
    order = np.argsort(x)
    x = x[order]
    ww = ww[order]
    cdf = np.cumsum(ww) / np.sum(ww)
    return float(np.interp(q, cdf, x))


def _condition_to_weights(
    da: xr.DataArray,
    weights: xr.DataArray,
    condition: xr.DataArray | None = None,
) -> xr.DataArray:
    """Broadcast spatial weights to `da` and apply an optional condition."""
    w = weights.broadcast_like(da)
    if condition is not None:
        condition = condition.broadcast_like(da)
        w = w.where(condition)
    return w


def weighted_masked_mean(
    da: xr.DataArray,
    weights: xr.DataArray,
    condition: xr.DataArray | None = None,
) -> float:
    """Weighted mean over all dimensions of `da`, excluding NaNs and zero-weight cells."""
    w = _condition_to_weights(da, weights, condition)
    valid = np.isfinite(da) & np.isfinite(w) & (w > 0)
    denom = w.where(valid).sum(skipna=True)
    denom_value = float(denom.values)
    if not np.isfinite(denom_value) or denom_value <= 0:
        return np.nan
    num = (da.where(valid).fillna(0.0) * w.where(valid).fillna(0.0)).sum(skipna=True)
    return float((num / denom).values)


def valid_cell_month_count(
    da: xr.DataArray,
    weights: xr.DataArray,
    condition: xr.DataArray | None = None,
) -> int:
    """Unweighted count of finite evaluated cell-months under the same mask as a metric."""
    w = _condition_to_weights(da, weights, condition)
    valid = np.isfinite(da) & np.isfinite(w) & (w > 0)
    return int(valid.sum().values)


def weighted_corr(
    pred: xr.DataArray,
    obs: xr.DataArray,
    weights: xr.DataArray,
    condition: xr.DataArray | None = None,
) -> float:
    """Weighted Pearson correlation over flattened cell-months."""
    pred, obs = xr.align(pred, obs, join="inner")
    w = _condition_to_weights(obs, weights, condition)

    x = np.asarray(pred.values, dtype="float64").ravel()
    y = np.asarray(obs.values, dtype="float64").ravel()
    ww = np.asarray(w.values, dtype="float64").ravel()
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(ww) & (ww > 0)
    x = x[valid]
    y = y[valid]
    ww = ww[valid]
    if x.size < 2 or np.sum(ww) <= 0:
        return np.nan

    ww = ww / np.sum(ww)
    mx = np.sum(ww * x)
    my = np.sum(ww * y)
    vx = np.sum(ww * (x - mx) ** 2)
    vy = np.sum(ww * (y - my) ** 2)
    if vx <= 0 or vy <= 0:
        return np.nan
    cov = np.sum(ww * (x - mx) * (y - my))
    return float(cov / np.sqrt(vx * vy))


def subset_conditions(observation: xr.DataArray, extreme_threshold: float) -> dict[str, xr.DataArray | None]:
    """Named evaluation subsets used everywhere in the project."""
    months = month_coord(observation)
    fire_season = months.isin(FIRE_SEASON_MONTHS).broadcast_like(observation)
    august = (months == 8).broadcast_like(observation)
    extreme = (observation >= extreme_threshold)
    return {
        "all": None,
        "fire_season_Jul-Oct": fire_season,
        "august": august,
        f"extreme_top_{int((1 - EXTREME_Q) * 100)}pct_train": extreme,
    }


def combine_conditions(*conditions: xr.DataArray | None) -> xr.DataArray | None:
    """Logical-AND non-null conditions, preserving xarray coordinates."""
    out = None
    for condition in conditions:
        if condition is None:
            continue
        condition = condition.astype(bool)
        out = condition if out is None else (out & condition)
    return out


def evaluate_forecast(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    baseline_name: str,
    split_name: str,
    extreme_threshold: float,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Evaluate one forecast against one target split.

    All reported quantities use the same valid-pair mask: finite forecast mean,
    finite observation, finite CRPS, positive metric weight, plus the requested
    named subset condition. This prevents missing persistence forecasts from
    silently contributing to counts, means, or reliability diagnostics.
    """
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True).clip(min=0.0, max=1.0).rename("forecast_mean")
    error = (forecast_mean - observation).rename("error")
    abs_error = np.abs(error).rename("abs_error")
    squared_error = (error ** 2).rename("squared_error")
    crps = crps_ensemble(forecast_ens, observation, member_dim=member_dim)
    valid_pair = (np.isfinite(forecast_mean) & np.isfinite(observation) & np.isfinite(crps)).rename("valid_pair")

    rows = []
    for subset_name, subset_condition in subset_conditions(observation, extreme_threshold).items():
        eval_condition = combine_conditions(subset_condition, valid_pair)
        mse = weighted_masked_mean(squared_error, weights, eval_condition)
        rows.append({
            "baseline": baseline_name,
            "split": split_name,
            "subset": subset_name,
            "n_cell_months": valid_cell_month_count(crps, weights, eval_condition),
            "obs_mean": weighted_masked_mean(observation, weights, eval_condition),
            "forecast_mean": weighted_masked_mean(forecast_mean, weights, eval_condition),
            "crps": weighted_masked_mean(crps, weights, eval_condition),
            "rmse": float(np.sqrt(mse)) if np.isfinite(mse) else np.nan,
            "mae": weighted_masked_mean(abs_error, weights, eval_condition),
            "bias": weighted_masked_mean(error, weights, eval_condition),
            "corr": weighted_corr(forecast_mean, observation, weights, eval_condition),
        })
    return pd.DataFrame(rows)


def add_skill_scores(metrics: pd.DataFrame, reference_baseline: str = "climatology") -> pd.DataFrame:
    """Add skill scores relative to the reference baseline for proper/loss metrics."""
    out = metrics.copy()
    key_cols = ["split", "subset"]
    for metric in ["crps", "rmse", "mae"]:
        ref = out[out["baseline"] == reference_baseline][key_cols + [metric]].rename(columns={metric: f"{metric}_reference"})
        out = out.merge(ref, on=key_cols, how="left")
        out[f"{metric}_skill_vs_{reference_baseline}"] = 1.0 - out[metric] / out[f"{metric}_reference"]
        out = out.drop(columns=[f"{metric}_reference"])
    return out


In [ ]:
extreme_threshold = weighted_quantile(targets["train"].where(land_mask), area_weights, EXTREME_Q)
print(f"Train weighted top-1% burned-fraction threshold: {extreme_threshold:.8g}")

metric_frames = []
for (baseline_name, split_name), forecast_ens in forecasts.items():
    metric_frames.append(
        evaluate_forecast(
            forecast_ens=forecast_ens,
            observation=targets[split_name],
            weights=area_weights,
            baseline_name=baseline_name,
            split_name=split_name,
            extreme_threshold=extreme_threshold,
        )
    )

metrics = add_skill_scores(pd.concat(metric_frames, ignore_index=True), reference_baseline="climatology")
metrics_path = METRIC_DIR / "trivial_baselines_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)
display(metrics)

## 6. Monthly scores and bootstrap uncertainty for CRPS

The headline metric is a weighted cell-month mean. For bootstrap confidence intervals, especially on conditional subsets such as the train top-1% extreme tail, we therefore resample months but recompute the weighted ratio:

\[
rac{\sum_t \sum_i w_i s_{t,i}}{\sum_t \sum_i w_i}.
\]

This avoids the bias from averaging monthly regional scores equally when different months contain very different numbers of evaluated extreme cells.


In [ ]:
def monthly_score_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    baseline_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Month-of-year score table for one forecast/split."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True).clip(min=0.0, max=1.0)
    crps = crps_ensemble(forecast_ens, observation, member_dim=member_dim)
    error = forecast_mean - observation
    valid_pair = (np.isfinite(forecast_mean) & np.isfinite(observation) & np.isfinite(crps)).rename("valid_pair")
    months = month_coord(observation)

    rows = []
    for month in range(1, 13):
        month_condition = (months == month).broadcast_like(observation)
        eval_condition = combine_conditions(month_condition, valid_pair)
        mse = weighted_masked_mean(error ** 2, weights, eval_condition)
        rows.append({
            "baseline": baseline_name,
            "split": split_name,
            "month": month,
            "n_cell_months": valid_cell_month_count(crps, weights, eval_condition),
            "obs_mean": weighted_masked_mean(observation, weights, eval_condition),
            "forecast_mean": weighted_masked_mean(forecast_mean, weights, eval_condition),
            "crps": weighted_masked_mean(crps, weights, eval_condition),
            "rmse": float(np.sqrt(mse)) if np.isfinite(mse) else np.nan,
            "mae": weighted_masked_mean(np.abs(error), weights, eval_condition),
            "bias": weighted_masked_mean(error, weights, eval_condition),
            "corr": weighted_corr(forecast_mean, observation, weights, eval_condition),
        })
    return pd.DataFrame(rows)


monthly_frames = []
for (baseline_name, split_name), forecast_ens in forecasts.items():
    monthly_frames.append(monthly_score_table(forecast_ens, targets[split_name], area_weights, baseline_name, split_name))
monthly_metrics = pd.concat(monthly_frames, ignore_index=True)
monthly_metrics_path = METRIC_DIR / "trivial_baselines_monthly_metrics.csv"
monthly_metrics.to_csv(monthly_metrics_path, index=False)

print("Saved:", monthly_metrics_path)
display(monthly_metrics.head(12))


In [ ]:
def monthly_weighted_score_components(
    score_da: xr.DataArray,
    weights: xr.DataArray,
    condition: xr.DataArray | None = None,
) -> pd.DataFrame:
    """Return one weighted numerator/denominator row per target month.

    The weighted score over any collection of rows is
    `sum(weighted_numerator) / sum(weighted_denominator)`. This is the same
    definition used by `weighted_masked_mean`, and is robust for conditional
    subsets with uneven spatial support across months.
    """
    rows = []
    for t in score_da.time.values:
        score_t = score_da.sel(time=t)
        cond_t = condition.sel(time=t) if condition is not None and "time" in condition.dims else condition
        w = _condition_to_weights(score_t, weights, cond_t)
        valid = np.isfinite(score_t) & np.isfinite(w) & (w > 0)
        denominator = float(w.where(valid).sum(skipna=True).values)
        if np.isfinite(denominator) and denominator > 0:
            numerator = float((score_t.where(valid).fillna(0.0) * w.where(valid).fillna(0.0)).sum(skipna=True).values)
            score = numerator / denominator
            n_cell_months = int(valid.sum().values)
        else:
            numerator = 0.0
            score = np.nan
            n_cell_months = 0
        rows.append({
            "time": pd.Timestamp(t),
            "weighted_numerator": numerator,
            "weighted_denominator": denominator,
            "score": score,
            "n_cell_months": n_cell_months,
        })
    return pd.DataFrame(rows).sort_values("time")


def bootstrap_weighted_ratio_ci(
    components: pd.DataFrame,
    n_boot: int = N_BOOT,
    alpha: float = BOOT_ALPHA,
    seed: int = RANDOM_SEED,
) -> dict:
    """Bootstrap CI for a weighted score by resampling months.

    This preserves the project metric definition for all subsets:
    `sum(weighted_numerator) / sum(weighted_denominator)`.
    """
    valid = components[
        np.isfinite(components["weighted_numerator"])
        & np.isfinite(components["weighted_denominator"])
        & (components["weighted_denominator"] > 0)
    ].copy()
    if valid.empty:
        return {
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_months": 0,
            "n_cell_months": 0,
            "weighted_denominator": 0.0,
        }

    numerator = valid["weighted_numerator"].to_numpy(dtype="float64")
    denominator = valid["weighted_denominator"].to_numpy(dtype="float64")
    n_months = numerator.size
    rng_local = np.random.default_rng(seed)
    idx = rng_local.integers(0, n_months, size=(n_boot, n_months))
    boot_den = denominator[idx].sum(axis=1)
    boot_num = numerator[idx].sum(axis=1)
    boot = np.divide(boot_num, boot_den, out=np.full(n_boot, np.nan), where=boot_den > 0)

    total_den = denominator.sum()
    mean = float(numerator.sum() / total_den) if total_den > 0 else np.nan
    return {
        "mean": mean,
        "ci_low": float(np.nanquantile(boot, alpha / 2)),
        "ci_high": float(np.nanquantile(boot, 1 - alpha / 2)),
        "n_months": int(n_months),
        "n_cell_months": int(valid["n_cell_months"].sum()),
        "weighted_denominator": float(total_den),
    }


def crps_bootstrap_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    baseline_name: str,
    split_name: str,
    extreme_threshold: float,
) -> pd.DataFrame:
    """Bootstrap uncertainty for weighted CRPS across named subsets."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean("member", skipna=True)
    crps = crps_ensemble(forecast_ens, observation)
    valid_pair = (np.isfinite(forecast_mean) & np.isfinite(observation) & np.isfinite(crps)).rename("valid_pair")

    rows = []
    for subset_name, subset_condition in subset_conditions(observation, extreme_threshold).items():
        eval_condition = combine_conditions(subset_condition, valid_pair)
        components = monthly_weighted_score_components(crps, weights, eval_condition)
        ci = bootstrap_weighted_ratio_ci(components)
        ci.update({"baseline": baseline_name, "split": split_name, "subset": subset_name})
        rows.append(ci)
    return pd.DataFrame(rows)


boot_frames = []
for (baseline_name, split_name), forecast_ens in forecasts.items():
    boot_frames.append(crps_bootstrap_table(forecast_ens, targets[split_name], area_weights, baseline_name, split_name, extreme_threshold))
crps_bootstrap = pd.concat(boot_frames, ignore_index=True)
crps_bootstrap_path = METRIC_DIR / "trivial_baselines_crps_bootstrap_ci.csv"
crps_bootstrap.to_csv(crps_bootstrap_path, index=False)

print("Saved:", crps_bootstrap_path)
display(crps_bootstrap)


## 7. Reliability table helper

The reliability helper is defined now so later probabilistic models can use it unchanged. For deterministic one-member baselines, event probabilities are only 0 or 1, so the reliability table is sparse; that is expected.

Missing forecasts must remain missing. In particular, the first persistence forecast in the full record is `NaN`, and this helper excludes it rather than turning it into a zero-probability event via `NaN > threshold == False`.


In [ ]:
def reliability_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    threshold: float,
    weights: xr.DataArray,
    baseline_name: str,
    split_name: str,
    n_bins: int = 10,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Weighted reliability table for event P(y > threshold)."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")

    # Preserve missing forecasts as NaN before thresholding. Otherwise
    # `NaN > threshold` becomes False and silently creates zero-probability events.
    forecast_event = xr.where(
        np.isfinite(forecast_ens),
        (forecast_ens > threshold).astype("float64"),
        np.nan,
    )
    prob = forecast_event.mean(member_dim, skipna=True).rename("forecast_probability")
    event = xr.where(
        np.isfinite(observation),
        (observation > threshold).astype("float64"),
        np.nan,
    ).rename("event")
    valid_pair = (np.isfinite(prob) & np.isfinite(event)).rename("valid_pair")
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    rows = []
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
        if i == n_bins - 1:
            bin_condition = (prob >= lo) & (prob <= hi)
        else:
            bin_condition = (prob >= lo) & (prob < hi)
        in_bin = combine_conditions(bin_condition, valid_pair)
        w = weights.broadcast_like(prob).where(in_bin)
        denom = float(w.sum(skipna=True).values)
        if denom <= 0 or not np.isfinite(denom):
            mean_prob = np.nan
            observed_frequency = np.nan
            n_cell_months = 0
        else:
            mean_prob = weighted_masked_mean(prob, weights, in_bin)
            observed_frequency = weighted_masked_mean(event, weights, in_bin)
            n_cell_months = valid_cell_month_count(prob, weights, in_bin)
        rows.append({
            "baseline": baseline_name,
            "split": split_name,
            "threshold": threshold,
            "bin": i,
            "bin_left": lo,
            "bin_right": hi,
            "mean_forecast_probability": mean_prob,
            "observed_frequency": observed_frequency,
            "weighted_bin_mass": denom,
            "n_cell_months": n_cell_months,
        })
    return pd.DataFrame(rows)


reliability_frames = []
for (baseline_name, split_name), forecast_ens in forecasts.items():
    reliability_frames.append(
        reliability_table(
            forecast_ens=forecast_ens,
            observation=targets[split_name],
            threshold=BURN_THRESHOLD,
            weights=area_weights,
            baseline_name=baseline_name,
            split_name=split_name,
            n_bins=10,
        )
    )
reliability = pd.concat(reliability_frames, ignore_index=True)
reliability_path = METRIC_DIR / "trivial_baselines_reliability_active_event.csv"
reliability.to_csv(reliability_path, index=False)

print("Saved:", reliability_path)
display(reliability[reliability["n_cell_months"] > 0].head(20))


## 8. Visual checks

In [ ]:
def plot_metric_bars(metrics_df: pd.DataFrame, split_name: str = "test", subset_name: str = "all") -> None:
    """Compact CRPS/RMSE comparison for a split/subset."""
    df = metrics_df[(metrics_df["split"] == split_name) & (metrics_df["subset"] == subset_name)].copy()
    if df.empty:
        print(f"No rows for split={split_name!r}, subset={subset_name!r}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
    df.set_index("baseline")["crps"].plot(kind="bar", ax=axes[0])
    axes[0].set_title(f"CRPS — {split_name}, {subset_name}")
    axes[0].set_ylabel("CRPS")
    axes[0].set_xlabel("")

    df.set_index("baseline")["rmse"].plot(kind="bar", ax=axes[1])
    axes[1].set_title(f"RMSE — {split_name}, {subset_name}")
    axes[1].set_ylabel("RMSE")
    axes[1].set_xlabel("")
    plt.tight_layout()
    plt.show()


plot_metric_bars(metrics, split_name="val", subset_name="all")
plot_metric_bars(metrics, split_name="test", subset_name="all")
plot_metric_bars(metrics, split_name="test", subset_name="fire_season_Jul-Oct")

In [ ]:
def plot_monthly_scores(monthly_df: pd.DataFrame, split_name: str = "test", metric: str = "crps") -> None:
    """Month-of-year score curves for a chosen split."""
    df = monthly_df[monthly_df["split"] == split_name]
    if df.empty:
        print(f"No monthly rows for split={split_name!r}")
        return

    fig, ax = plt.subplots(figsize=(9, 4.8))
    for baseline_name, group in df.groupby("baseline"):
        group = group.sort_values("month")
        ax.plot(group["month"], group[metric], marker="o", label=baseline_name)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Target month")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"Monthly {metric.upper()} — {split_name}")
    ax.legend()
    plt.show()


plot_monthly_scores(monthly_metrics, split_name="val", metric="crps")
plot_monthly_scores(monthly_metrics, split_name="test", metric="crps")
plot_monthly_scores(monthly_metrics, split_name="test", metric="rmse")

In [ ]:
def plot_reliability(rel_df: pd.DataFrame, baseline_name: str, split_name: str) -> None:
    """Reliability diagram from a precomputed reliability table."""
    df = rel_df[
        (rel_df["baseline"] == baseline_name)
        & (rel_df["split"] == split_name)
        & (rel_df["n_cell_months"] > 0)
    ].copy()
    if df.empty:
        print(f"No non-empty reliability bins for {baseline_name}/{split_name}")
        return

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot([0, 1], [0, 1], linestyle="--", label="perfect reliability")
    ax.plot(df["mean_forecast_probability"], df["observed_frequency"], marker="o", label=baseline_name)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean forecast probability")
    ax.set_ylabel("Observed event frequency")
    ax.set_title(f"Reliability: active burning event — {split_name}")
    ax.legend()
    plt.show()


plot_reliability(reliability, "climatology", "test")
plot_reliability(reliability, "persistence", "test")

## 9. Case-study hooks: 2017 Portugal and 2022 Iberian summer

In [ ]:
def _finite_max_abs(da: xr.DataArray) -> float:
    values = np.asarray(da.values, dtype="float64")
    if not np.isfinite(values).any():
        return np.nan
    return float(np.nanmax(np.abs(values)))


def _finite_max(da: xr.DataArray) -> float:
    values = np.asarray(da.values, dtype="float64")
    if not np.isfinite(values).any():
        return np.nan
    return float(np.nanmax(values))


def plot_map_triplet(observed: xr.DataArray, forecast: xr.DataArray, title: str) -> None:
    """Observed, forecast, and signed-error maps for one month.

    Observed and forecast panels share the same burned-fraction colour scale so
    trivial-baseline underprediction is visually honest in extreme months.
    """
    observed = observed.where(land_mask)
    forecast = forecast.where(land_mask)
    error = (forecast - observed).where(land_mask)

    burn_vmax = np.nanmax([_finite_max(observed), _finite_max(forecast)])
    if not np.isfinite(burn_vmax) or burn_vmax <= 0:
        burn_vmax = 1e-6
    error_vmax = _finite_max_abs(error)
    if not np.isfinite(error_vmax) or error_vmax <= 0:
        error_vmax = burn_vmax

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    observed.plot(ax=axes[0], vmin=0.0, vmax=burn_vmax, cbar_kwargs={"label": "burned fraction"})
    axes[0].set_title("Observed")
    forecast.plot(ax=axes[1], vmin=0.0, vmax=burn_vmax, cbar_kwargs={"label": "burned fraction"})
    axes[1].set_title("Forecast mean")
    error.plot(ax=axes[2], vmin=-error_vmax, vmax=error_vmax, cbar_kwargs={"label": "forecast - observed"})
    axes[2].set_title("Signed error")
    for ax in axes:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_case_study_month(baseline_name: str, date_str: str) -> None:
    """Plot a case-study month if it exists in any split."""
    date = pd.Timestamp(date_str)
    matching_split = None
    for split_name, target in targets.items():
        if date in set(pd.DatetimeIndex(target.time.values)):
            matching_split = split_name
            break
    if matching_split is None:
        print(f"Skipping {date_str}: not present in processed targets")
        return

    forecast_ens = forecasts[(baseline_name, matching_split)]
    forecast_mean = forecast_ens.mean("member", skipna=True).sel(time=date)
    observed = targets[matching_split].sel(time=date)
    plot_map_triplet(observed, forecast_mean, f"{baseline_name} baseline — {date:%Y-%m} ({matching_split})")


case_study_months = [
    "2017-06-01",
    "2017-08-01",
    "2017-10-01",
    "2022-07-01",
    "2022-08-01",
]

for baseline_name in ["climatology", "persistence"]:
    for date_str in case_study_months:
        plot_case_study_month(baseline_name, date_str)


## 10. Locked artefacts written by this notebook

In [ ]:
artefacts = pd.DataFrame([
    {"kind": "baseline", "path": str(BASELINE_DIR / "climatology_monthly_mean_train_only.nc")},
    *[
        {"kind": "forecast", "path": str(BASELINE_DIR / f"forecast_{baseline}_{split}.nc")}
        for baseline in ["climatology", "persistence"]
        for split in ["train", "val", "test"]
    ],
    {"kind": "metrics", "path": str(METRIC_DIR / "trivial_baselines_metrics.csv")},
    {"kind": "metrics", "path": str(METRIC_DIR / "trivial_baselines_monthly_metrics.csv")},
    {"kind": "metrics", "path": str(METRIC_DIR / "trivial_baselines_crps_bootstrap_ci.csv")},
    {"kind": "metrics", "path": str(METRIC_DIR / "trivial_baselines_reliability_active_event.csv")},
])

display(artefacts)